In [1]:
%pip install numba

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 9.2 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 389.2 kB/s  0:01:20m0:00:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [numba]32m1/2 [numba]
Note: you may need to restart the kernel to use updated packages.


In [55]:
import numpy as np
from numba import njit
import time

# --- 1. The Matrix Packer ---
@njit
def pack_binary_matrix(binary_matrix):
    """
    Compresses an (N x M) binary matrix into an (N x W) uint64 matrix.
    Every 64 columns of the original matrix become 1 column of 64-bit integers.
    """
    rows, cols = binary_matrix.shape
    # Calculate how many 64-bit words we need per row (ceiling division)
    words = (cols + 63) // 64 
    
    # Initialize the compressed matrix with 64-bit unsigned integers
    packed = np.zeros((rows, words), dtype=np.uint64)
    
    for r in range(rows):
        for c in range(cols):
            if binary_matrix[r, c] == 1:
                word_idx = c // 64
                bit_idx = c % 64
                # Force the 1 to be a 64-bit type before shifting to prevent overflow
                bit_mask = np.uint64(1) << np.uint64(bit_idx)
                packed[r, word_idx] |= bit_mask
                
    return packed

# --- 2. The Ultra-Fast Rank Solver ---
@njit
def fast_packed_gf2_rank(packed_matrix, num_cols):
    """
    Computes the GF(2) rank directly on the packed uint64 memory blocks.
    Utilizes the upper-triangular shortcut to cut mathematical operations in half.
    """
    mat = packed_matrix.copy()
    rows, words = mat.shape
    rank = 0
    
    for col in range(num_cols):
        # Locate the exact integer and the exact bit within it for the current column
        word_idx = col // 64
        bit_idx = col % 64
        
        # 1. Pivot search (Optimized: Only search from the current rank downwards)
        pivot_row = -1
        for r in range(rank, rows):
            # Shift the target bit to the 0th position and mask it with 1
            if (mat[r, word_idx] >> np.uint64(bit_idx)) & np.uint64(1):
                pivot_row = r
                break
                
        if pivot_row == -1:
            continue
            
        # 2. Row swap
        if pivot_row != rank:
            for w in range(words):
                temp = mat[rank, w]
                mat[rank, w] = mat[pivot_row, w]
                mat[pivot_row, w] = temp
                
        # 3. Row elimination via XOR 
        # (Optimized: Only eliminate strictly BELOW the pivot to form Row Echelon Form)
        for r in range(rank + 1, rows):
            if (mat[r, word_idx] >> np.uint64(bit_idx)) & np.uint64(1):
                # The Hardware Magic: XORing a single word processes 64 bits simultaneously
                for w in range(words):
                    mat[r, w] ^= mat[rank, w]
                    
        rank += 1
        if rank == rows:
            break
            
    return rank

# ==========================================
# BENCHMARK TEST (10000 x 10000)
# ==========================================
if __name__ == "__main__":
    N = 20000
    print(f"Generating random {N} x {N} binary matrix...")
    # Use uint8 for the initial generation to save RAM before packing
    binary_matrix = np.random.randint(0, 2, (N, N), dtype=np.uint8)
    
    # Pack the matrix
    t0 = time.time()
    packed_mat = pack_binary_matrix(binary_matrix)
    t1 = time.time()
    
    print(f"Matrix packed into {packed_mat.shape[1]} words per row in {t1 - t0:.4f} seconds.")
    
    # Calculate the rank
    # Run once to trigger the Numba JIT compiler (warm-up)
    fast_packed_gf2_rank(packed_mat[:10, :], 10) 
    
    t2 = time.time()
    rank = fast_packed_gf2_rank(packed_mat, num_cols=N)
    t3 = time.time()
    
    print(f"Rank: {rank}")
    print(f"Rank calculation time: {t3 - t2:.4f} seconds.")

Generating random 20000 x 20000 binary matrix...
Matrix packed into 313 words per row in 1.4658 seconds.
Rank: 19937
Rank calculation time: 14.0229 seconds.


Let's start with a simple system.